In [2]:
import pandas as pd
DATA = 'data/sample_diverse_10_ux_agent_3.csv'
sims = pd.read_csv(DATA)
sims

,starting_url,run_folder,persona_name,persona_description,flow_name,flow_description,category,name
0,https://www.nationwide.co.uk/,runs/2025-12-03_15-36-10_0913,Young professional user,You are a young professional in your mid-20s. ...,Get customer support number,Get the number for customer service. Stop once...,financial,Sim 118
1,https://www.hsbc.co.uk/,runs/2025-12-03_15-42-09_cad1,Experienced user,You are an experienced user who has used this ...,Get customer support number,Get the number for customer service. Stop once...,financial,Sim 76
2,https://www.barclays.co.uk/,runs/2025-12-03_15-45-50_684b,First-time user,You are a first-time user with no prior experi...,Get customer support number,Get the number for customer service. Stop once...,financial,Sim 64
3,https://www.barclays.co.uk/,runs/2025-12-03_15-53-29_aaf1,Generic user,A generic user.,Locate a nearby branch,Find a branch that's near WC1E 6AE. Stop once ...,financial,Sim 68
4,https://www.hsbc.co.uk/,runs/2025-12-03_16-00-31_3092,Generic user,A generic user.,Get customer support number,Get the number for customer service. Stop once...,financial,Sim 82
5,https://www.primark.com/,runs/2025-12-03_16-24-57_495f,Generic user,A generic user.,Find ladies cotton trousers on sale,Find a pair of trousers or joggers that contai...,fashion,Sim 36
6,https://www.next.co.uk/,runs/2025-12-03_16-55-29_75de,Older user,You are a retiree in their 70s. You have typic...,Find ladies cotton trousers on sale,Find a pair of trousers or joggers that contai...,fashion,Sim 9
7,https://www.nike.com/gb/,runs/2025-12-03_17-45-04_25d8,First-time user,You are a first-time user with no prior experi...,Find men's trainers,Find a pair of trainers in men's size 10 in a ...,fashion,Sim 19
8,https://www.barclays.co.uk/,runs/2025-12-03_18-01-45_9b47,Older user,You are a retiree in their 70s. You have typic...,Find a credit card,Find a credit card that suits your needs. Stop...,financial,Sim 69
9,https://www.zara.com/uk/,runs/2025-12-03_18-10-49_1525,Older user,You are a retiree in their 70s. You have typic...,Find men's trainers,Find a pair of trainers in men's size 10 in a ...,fashion,Sim 55


In [3]:
import json
import os
import anthropic

client = anthropic.Anthropic()

def count_tokens(text):
    response = client.messages.count_tokens(
    model="claude-sonnet-4-5",
    messages=[{
        "role": "user",
        "content": text
    }],
)

    return json.loads(response.model_dump_json())['input_tokens']

def get_text_from_run(run_folder):
    # get the action_trace.json, basic_info.json, and memory.json files from the run folder
    # concat it
    # return the text
    with open(run_folder + '/action_trace.json', 'r') as f:
        action_trace = json.load(f)
    with open(run_folder + '/basic_info.json', 'r') as f:
        basic_info = json.load(f)
    with open(run_folder + '/memory_trace.json', 'r') as f:
        memory = json.load(f)

    # get the simplified HTML from all the files in the run_folder/simp_html folder
    simplified_html = ''
    for file in os.listdir(run_folder + '/simp_html'):
        with open(run_folder + '/simp_html/' + file, 'r') as f:
            simplified_html += f.read()

    output= str(action_trace) + str(basic_info) + str(memory) + str(simplified_html)
    
    print(f'{run_folder}: {count_tokens(output)}')
    
    return output
    
sims['text'] = sims.apply(lambda x: get_text_from_run(x['run_folder']), axis=1)

sims['text_len'] = sims['text'].apply(lambda x: len(x))
sims['input_tokens'] = sims['text'].apply(lambda x: count_tokens(x))
# 

runs/2025-12-03_15-36-10_0913: 57254
runs/2025-12-03_15-42-09_cad1: 32610
runs/2025-12-03_15-45-50_684b: 72191
runs/2025-12-03_15-53-29_aaf1: 54754
runs/2025-12-03_16-00-31_3092: 79342
runs/2025-12-03_16-24-57_495f: 386634
runs/2025-12-03_16-55-29_75de: 1214687
runs/2025-12-03_17-45-04_25d8: 247381
runs/2025-12-03_18-01-45_9b47: 90495
runs/2025-12-03_18-10-49_1525: 67434


In [4]:

# count the tokens in this html file "runs/2025-12-03_16-55-29_75de/raw_html/raw_html_0.html"
with open("runs/2025-12-03_16-55-29_75de/raw_html/raw_html_0.html", "r") as f:
    html = f.read()
    print(count_tokens(html))
    
with open("runs/2025-12-03_16-55-29_75de/memory_trace.json", "r") as f:
    memory_trace = f.read()
    print(count_tokens(memory_trace))

369303
139106


In [5]:
count_tokens(""""
<SYSTEM_CAPABILITY> 
* You are a UI/UX researcher who is reviewing a user's experience with a website. You are given a list of past messages and screenshots from the user.
* Markdown the user simulation in 1-2 sentences; concisely say what the user did and what they felt with all the important bits (e.g. "User did xyz.").
* Consider if they completed the ENTIRE task/user flow they were given.
* output a json with summary and success (ie if they completed the flow successfully or not)
* Be concise. Up to 2 sentences for the summary and 1-3 bullet points for the pain points and actionable improvements.
</SYSTEM_CAPABILITY> 
<IMPORTANT> 
Output in the following way.
{{
    "summary": "1-2 sentence summary: explain the flow of the users (ie navigated from x to y to z) and mention any friction points they faced.",
    "success": True or False,
    "recommendations": {{
    "product": "Return a list of up to 3 JSONs. Include a concise explanation of a relevant issue and recommendation, and a short 2–3 word description of a UX-level recommendation. Focus on **strategic, product-level observations** that connect directly to the user’s goals, needs, and frustrations during this simulation — not speculative or sweeping redesigns. These should concern the product’s value proposition, content clarity, feature completeness, or business relevance. Your suggestions should answer **'WHAT should we improve or clarify in the offering?'** or **'WHY might this product not yet meet the user's intent?'**, not **'HOW should the UI change?'** or **'WHAT new audience should we serve?'**.  Be realistic and proportionate: do **not** recommend major product pivots, adding entire new verticals, or front-page changes based on a single user flow. Instead, ground suggestions in observed user evidence .  Format: [{{'description':'<Description 1>','recommendation':'<Emoji + Recommendation 1>','priority':'high|medium|low'}},{{'description':'<Description 2>','recommendation':'<Emoji + Recommendation 2>','priority':'high|medium|low'}}]",
    "ux": "Return a list of up to 3 JSONs. Include a concise explanation of a relevant issue and recommendation, and a short 2–3 word description of a UX-level recommendation. Focus only on user journey, flow logic, decision-making clarity, emotional friction (e.g. based on [emotion] expressed), confidence, or cognitive load. Format: [{{'description':'<Description 1>','recommendation':'<Emoji + Recommendation 1>','priority':'high|medium|low'}},{{'description':'<Description 2>','recommendation':'<Emoji + Recommendation 2>','priority':'high|medium|low'}}]",
    "ui": "Return a list of up to 3 JSONs. Include a concise explanation of a relevant issue and recommendation, and a short 2–3 word description of a UI-level recommendation.  Focus only on layout, visual hierarchy, input behavior, responsiveness, spacing, or styling. Format: [{{'description':'<Description 1>','recommendation':'<Emoji + Recommendation 1>','priority':'high|medium|low'}},{{'description':'<Description 2>','recommendation':'<Emoji + Recommendation 2>','priority':'high|medium|low'}}]"
    }}
}}
Be extremely accurate and always make reference to real user steps. 
Make sure the recommendations are really useful, EXTREMELY DIVERSE and appropriate and helpful for the website given the flow and user. They should be highly realistic and reasonable.
Do not recommend removing cookie pop-ups as they are legally required.
Use (professional) emojis in the recommendations json keys eg ✅ ❌ ⚠️ for statuses 🧭 🔍 🧱 for usability recommendations types 🏠 → 🧾 → ✅ for flow paths.
</IMPORTANT>
""")

942

In [6]:
sims.to_csv('data/sample_uxagent_runs_4.csv', index=False)